# ACT + task ID on dataset v2 (400 demonstrations), Kaggle

Trains ACT on `Kimaa6/icub_gardener_v2` with the split `configs/act_split_v2.json`: 360 training and 40
validation episodes (the 20 v1 validation episodes kept, plus 20 new ones). **The only change from the first
policy is the data:** same 80k steps, batch 8, mixed precision, seed 1000, chunk 100. (The GPU differs: a T4 here,
the laptop's RTX 3050 for v1.)

**Before running:**
1. `gardener_v2` is on the Hugging Face Hub, private (on the laptop:
   `scripts/push_dataset.py --root data/lerobot/gardener_v2 --repo-id Kimaa6/icub_gardener_v2`).
2. `configs/act_split_v2.json` is committed and on `main`.
3. Notebook settings: **Accelerator GPU T4 x1**, **Internet on**, and a secret named **`HF_TOKEN`** (Add-ons, Secrets)
   with read access to the dataset.
4. Run it with **Save Version, Save & Run All (Commit)**: it then runs on its own, up to 12 h, with the browser closed.

**Output** (Output tab): `act_v2_best.zip` with the 3 checkpoints of lowest validation error, the training log
and the validation table. Every checkpoint is also kept in `/kaggle/working/act_v2/checkpoints/`.

In [ ]:
# 1. GPU and versions
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version

In [ ]:
# 2. Code: the public repo, main branch. The commit is printed so the run can be traced.
!rm -rf /kaggle/working/icub-gardener
!git clone -q -b main https://github.com/kaarima/icub-gardener.git /kaggle/working/icub-gardener
!cd /kaggle/working/icub-gardener && git log --oneline -1

In [ ]:
# 3. LeRobot, the same version as on the laptop
!pip install -q lerobot==0.4.4
!python -c "import lerobot, torch; print('lerobot', lerobot.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

In [ ]:
# 4. Dataset v2 from the Hub (private: needs the HF_TOKEN secret)
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!hf download Kimaa6/icub_gardener_v2 --repo-type dataset --local-dir /kaggle/working/gardener_v2 --quiet
!du -sh /kaggle/working/gardener_v2

In [ ]:
# 5. Checks before spending GPU hours: 400 episodes, the agreed split, and one frame decodes.
import json
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

REPO, ROOT = "Kimaa6/icub_gardener_v2", "/kaggle/working/gardener_v2"
split = json.load(open("/kaggle/working/icub-gardener/configs/act_split_v2.json"))
v1 = json.load(open("/kaggle/working/icub-gardener/configs/act_split.json"))
meta = LeRobotDatasetMetadata(REPO, root=ROOT)
train, val = set(split["train"]), set(split["val"])
assert split["dataset"] == REPO, split["dataset"]
assert meta.total_episodes == 400, meta.total_episodes
assert len(train) == 360 and len(val) == 40, (len(train), len(val))
assert not train & val, "an episode is in both training and validation"
assert train | val == set(range(400)), "an episode is missing"
assert set(v1["val"]) <= val, "a v1 validation episode is not in validation"
assert all(e >= 200 for e in val - set(v1["val"])), "a new validation episode is not a new recording"
item = LeRobotDataset(REPO, root=ROOT, episodes=[split["val"][0]])[0]
print({k: tuple(v.shape) for k, v in item.items() if hasattr(v, "shape")})
print(f"OK: {meta.total_episodes} episodes, {meta.total_frames} frames; 360 train / 40 val, none in both, 20 v1 val kept")

In [ ]:
# 6. Train: 80k steps, batch 8, a checkpoint every 10k. The log is also saved to a file.
!cd /kaggle/working/icub-gardener && DATASET=/kaggle/working/gardener_v2 REPO_ID=Kimaa6/icub_gardener_v2 \
   SPLIT=configs/act_split_v2.json OUT=/kaggle/working/act_v2 STEPS=80000 BATCH=8 SAVE=10000 WORKERS=4 \
   bash scripts/train_act.sh 2>&1 | tee /kaggle/working/act_v2_train.log | grep -E "ACT:|step:|Checkpoint|Error|error" | awk 'NR % 10 == 1'

If a session stops before 80k, start a new one, run cells 1-5, then resume instead of cell 6 (after attaching the
earlier output as an input and copying `act_v2` back to `/kaggle/working/`):

`!lerobot-train --config_path=/kaggle/working/act_v2/checkpoints/last/pretrained_model/train_config.json --resume=true`

In [ ]:
# 7. Validation and training error for every checkpoint, on the v2 split
!cd /kaggle/working/icub-gardener && python scripts/val_loss.py /kaggle/working/act_v2/checkpoints/0*/pretrained_model \
   --root /kaggle/working/gardener_v2 --repo-id Kimaa6/icub_gardener_v2 --split configs/act_split_v2.json \
   2>&1 | grep -v Warn | tee /kaggle/working/act_v2_val.txt

In [ ]:
# 8. Package the 3 checkpoints with the lowest validation error, for the dev-seed check on the laptop
import re, subprocess
rows = re.findall(r"checkpoints/(\d+)/pretrained_model: validation ([0-9.]+)", open("/kaggle/working/act_v2_val.txt").read())
best = sorted(rows, key=lambda r: float(r[1]))[:3]
print("lowest validation error:", best)
paths = [f"act_v2/checkpoints/{s}/pretrained_model" for s, _ in best] + ["act_v2_train.log", "act_v2_val.txt"]
subprocess.run(["zip", "-qr", "act_v2_best.zip", *paths], cwd="/kaggle/working", check=True)
!ls -lh /kaggle/working/act_v2_best.zip